<a href="https://colab.research.google.com/github/Ayseatci/DI725_Project/blob/dev/notebooks/03_phase2/SWIN_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Image Model: SWIN V2

In [1]:
!pip install -q timm transformers accelerate

In [ ]:
import os
import shutil
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import mean_absolute_error, mean_squared_error

from torchvision import transforms
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip -q /content/drive/MyDrive/DI725_project_dataset_nomask.zip

In [ ]:
LOCAL_DATA_ROOT = "/content/DI725_project_dataset_nomask"
LOCAL_CSV_PATH = os.path.join(LOCAL_DATA_ROOT, "captions.csv")
LOCAL_IMAGE_DIR = os.path.join(LOCAL_DATA_ROOT, "images")

os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

CSV_PATH = LOCAL_CSV_PATH
IMAGE_DIR = LOCAL_IMAGE_DIR

print("CSV:", CSV_PATH)
print("Images:", IMAGE_DIR)
print("Number of images:", len(os.listdir(IMAGE_DIR)))

In [ ]:
!pip install -q wandb
import wandb

In [ ]:
CONFIG = {
    "sample_size": 10000,
    "img_size": 256,

    "batch_size": 32,
    "epochs": 15,
    "lr": 2e-4,
    "weight_decay": 1e-4,

    "model_name": "swinv2_tiny_window8_256",

    "early_stopping_patience": 3,
    "grad_clip": 1.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 42
}

TARGET_COLS = ["Tree", "Shrub", "Grass", "Crop", "Built-up", "Barren", "Water"]

In [ ]:
import re

TEXT_COLUMNS_RAW = [
    "hybrid_gemma3-4b",
    "text_qwen3-4b",
    "vision_gemma3-4b"
]

def clean_caption_no_numbers(text):
    text = str(text).lower()

    # remove percentages: 53%, 53 percent, 53 percentage
    text = re.sub(r"\b\d+(\.\d+)?\s*%|\b\d+(\.\d+)?\s*percent(age)?", "", text)

    # remove standalone numbers
    text = re.sub(r"\b\d+(\.\d+)?\b", "", text)

    leakage_words = [
        "covering", "coverage", "accounts for", "occupies",
        "making up", "comprises", "constitutes", "represents",
        "estimated", "approximately", "around", "about"
    ]

    for word in leakage_words:
        text = text.replace(word, "")

    text = re.sub(r"[%(),:;]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


for col in TEXT_COLUMNS_RAW:
    clean_col = col.replace("-", "_") + "_clean"

    df[clean_col] = df[col].apply(clean_caption_no_numbers)
    train_df[clean_col] = train_df[col].apply(clean_caption_no_numbers)
    val_df[clean_col] = val_df[col].apply(clean_caption_no_numbers)
    test_df[clean_col] = test_df[col].apply(clean_caption_no_numbers)

    print(clean_col)
    print("Remaining numeric leakage:",
          df[clean_col].str.contains(
              r"\d+\s*%|\d+\s*percent|\b\d+(\.\d+)?\b",
              regex=True,
              case=False
          ).sum())

In [ ]:
CONFIG["model_name"] = "swinv2_tiny_window8_256"
CONFIG["img_size"] = 256
CONFIG["epochs"] = 15
CONFIG["lr"] = 1e-4
CONFIG["batch_size"] = 32
CONFIG["text_scale"] = 0.7

IMG_SIZE = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
wandb.login()

REMOTECLIP_TEXT_COLUMNS = [
    "hybrid_gemma3_4b_clean",
    "text_qwen3_4b_clean",
    "vision_gemma3_4b_clean"
]

remoteclip_results = []

for text_col in REMOTECLIP_TEXT_COLUMNS:
    print("\n==============================")
    print("Running:", text_col)
    print("==============================")

    CONFIG["text_column"] = text_col

    wandb.init(
        project="DI725-landcover-regression",
        name=f"swinv2_remoteclip_{text_col}_scale_{CONFIG['text_scale']}",
        config=CONFIG,
        reinit=True
    )

    train_ds_remoteclip = LandCoverRawTextDataset(
        train_df,
        IMAGE_DIR,
        transform=train_tfms,
        text_col=CONFIG["text_column"]
    )

    val_ds_remoteclip = LandCoverRawTextDataset(
        val_df,
        IMAGE_DIR,
        transform=eval_tfms,
        text_col=CONFIG["text_column"]
    )

    test_ds_remoteclip = LandCoverRawTextDataset(
        test_df,
        IMAGE_DIR,
        transform=eval_tfms,
        text_col=CONFIG["text_column"]
    )

    train_loader_remoteclip = DataLoader(
        train_ds_remoteclip,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader_remoteclip = DataLoader(
        val_ds_remoteclip,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    test_loader_remoteclip = DataLoader(
        test_ds_remoteclip,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model_swin_remoteclip = SwinV2RemoteCLIPFusion(
        image_model_name=CONFIG["model_name"],
        num_outputs=len(TARGET_COLS),
        text_scale=CONFIG["text_scale"],
        unfreeze_last_text_block=False
    ).to(CONFIG["device"])

    model_swin_remoteclip, history_swin_remoteclip = train_remoteclip_model(
        model_swin_remoteclip,
        train_loader_remoteclip,
        val_loader_remoteclip,
        CONFIG
    )

    test_loss, test_mae, test_rmse, class_mae = evaluate_remoteclip_model(
        model_swin_remoteclip,
        test_loader_remoteclip
    )

    print("Text column:", text_col)
    print("Test MAE:", test_mae)
    print("Test RMSE:", test_rmse)

    wandb.log({
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_loss": test_loss
    })

    class_mae_table = wandb.Table(
        dataframe=pd.DataFrame({
            "class": TARGET_COLS,
            "class_mae": class_mae
        })
    )

    wandb.log({"class_mae_table": class_mae_table})

    remoteclip_results.append({
        "experiment": f"Swin V2 + RemoteCLIP + {text_col}",
        "model_name": CONFIG["model_name"],
        "text_model": "RemoteCLIP ViT-B-32",
        "text_column": text_col,
        "text_scale": CONFIG["text_scale"],
        "epochs": CONFIG["epochs"],
        "lr": CONFIG["lr"],
        "test_mae": test_mae,
        "test_rmse": test_rmse
    })

    wandb.finish()

remoteclip_results_df = pd.DataFrame(remoteclip_results)
remoteclip_results_df.sort_values("test_mae")